In [3]:
import torch
from d2l import torch as d2l
import numpy as np

In [4]:
def train_model(model, train_loader, val_loader, epochs=20, lr=1e-4,
                device=None, label_smoothing=0.1, save_path='best_model.pth'):
    if device is None:
        device = next(model.parameters()).device

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad,
               model.parameters()),
                lr=lr,
                weight_decay=5e-4  # 权重衰减
    )
    # 学习率调节器：调整学习率的 ########################################
    scheduler =ReduceLROnPlateau(optimizer, mode='min', factor=0)

    train_accs, val_accs = [], []
    best_val_acc = 0
    best_model_state = None

    print(f"\n{'=' * 60}")
    print(f"开始训练 (Epochs={epochs}, LR={lr}, Label Smoothing={label_smoothing})")
    print(f"{'=' * 60}\n")

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_acc = evaluate_accuracy(model, train_loader, device)
        val_acc = evaluate_accuracy(model, val_loader, device)
        train_accs.append(train_acc)
        val_accs.append(val_acc)


        avg_loss = train_loss / len(train_loader)
        print(f'Epoch {epoch + 1:3d}/{epochs}: Loss {avg_loss:.4f}, '
              f'Train Acc {train_acc:.4f}, Val Acc {val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            torch.save(best_model_state, save_path)
            print(f'  ✅ 保存最佳模型 (Val Acc: {best_val_acc:.4f})')

        scheduler.step(val_acc) # 在这根据验证指标自动调整，最常用 #####################################

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f'\n✅ 训练完成！最佳验证准确率: {best_val_acc:.4f}')
        print(f'   模型已保存到: {save_path}')

    return train_accs, val_accs, best_val_acc

In [5]:
# 注意事项
# 1 在评估后调用
# 2 scheduler.step() ，其中要传入验证指标

In [6]:
# 实例
def train_model(...):
    # 1. 创建优化器（管理权重更新）
    optimizer = optim.Adam(...)
    
    # 2. 创建 scheduler（管理学习率）
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='max',      # 最大化验证准确率
        factor=0.5,      # 每次减半
        patience=5,      # 5轮不提升就降低
        verbose=True,
        min_lr=1e-7
    )
    
    for epoch in range(epochs):
        # 3. 训练（更新权重）
        for batch in train_loader:
            loss = ...
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()  # ← 更新权重
        
        # 4. 评估
        val_acc = evaluate(...)
        
        # 5. 调整学习率（基于验证结果）
        scheduler.step(val_acc)  # ← 如果 val_acc 停滞，降低学习率
        
        # 6. 获取当前学习率（用于监控）
        current_lr = optimizer.param_groups[0]['lr']

SyntaxError: invalid syntax (2427062006.py, line 2)

In [ ]:
#常用的几种包括：
# StepLR是每隔固定轮数突然降低学习率，适合你有经验知道什么时候该降；
# MultiStepLR类似但可以指定在哪些具体轮次降低，更灵活；
# ExponentialLR则是每个轮次都按固定比例平滑下降，不依赖模型表现；
# ReduceLROnPlateau是最推荐的自适应方式，它会监控验证准确率，当连续多个轮次没有提升时就自动降低学习率，完全不需要你提前判断时机；
# CosineAnnealingLR按余弦曲线平滑下降，适合需要周期性调整的任务；
# CyclicLR让学习率在范围内循环震荡，用于特殊场景。
#对于你的图像分类任务，直接选ReduceLROnPlateau最省心，
#设置好patience（等待轮数）和factor（衰减比例）就行，它会根据训练过程自动决定何时降低学习率，帮助模型稳定收敛。